<a href="https://colab.research.google.com/github/jayaadityabalaji/company-x-slow-stock-review/blob/main/Company_X_Technology_and_Data_Management_Case_Study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip -q install pandas openpyxl
print("Libraries installed successfully.")


Libraries installed successfully.


In [2]:

from google.colab import files
import pandas as pd
import numpy as np
import re
from IPython.display import display, Markdown

uploaded = files.upload()

file_path = next(iter(uploaded.keys()))

xls = pd.ExcelFile(file_path)

required_sheets = [
    "Item_Master",
    "Stock_Aging",
    "Sales_Lines",
    "Customers"
]

missing = [s for s in required_sheets if s not in xls.sheet_names]

if missing:
    raise ValueError(f"Missing sheets: {missing}")

items = pd.read_excel(file_path, sheet_name="Item_Master")
stock = pd.read_excel(file_path, sheet_name="Stock_Aging")
sales = pd.read_excel(file_path, sheet_name="Sales_Lines")
customers = pd.read_excel(file_path, sheet_name="Customers")

print("Workbook loaded successfully.")
print("Item Master:", items.shape)
print("Stock Aging:", stock.shape)
print("Sales Lines:", sales.shape)
print("Customers:", customers.shape)

for name, df in [
    ("Item_Master", items),
    ("Stock_Aging", stock),
    ("Sales_Lines", sales),
    ("Customers", customers)
]:
    print(f"\n{name} columns:")
    print(df.columns.tolist())
    display(df.head(3))


Saving Case_Data_Technology_and_Data_Management_All_Sheets.xlsx to Case_Data_Technology_and_Data_Management_All_Sheets.xlsx
Workbook loaded successfully.
Item Master: (42, 8)
Stock Aging: (42, 6)
Sales Lines: (630, 7)
Customers: (30, 4)

Item_Master columns:
['SKU', 'Item Name', 'Category', 'Master Pack Qty', 'Unit Cost (Rs)', 'Alternate Item (name)', 'Alternate Flag', 'Status']


,SKU,Item Name,Category,Master Pack Qty,Unit Cost (Rs),Alternate Item (name),Alternate Flag,Status
0,SKU1001,A-Product 01 (50/case),Category A,50,25,NaN,NaN,Active
1,SKU1002,A-Product 02 (10/case),Category A,10,85,NaN,NaN,Active
2,SKU1003,A-Product 03 (12/case),Category A,12,60,A-Product 06 (10/case),SWITCH,Active



Stock_Aging columns:
['SKU', 'Qty 0-30 days', 'Qty 31-60 days', 'Qty 61-90 days', 'Qty 91-180 days', 'Qty 180+ days']


,SKU,Qty 0-30 days,Qty 31-60 days,Qty 61-90 days,Qty 91-180 days,Qty 180+ days
0,SKU1001,175,100,75,20,60
1,SKU1002,13,7,6,7,4
2,SKU1003,45,26,19,23,16



Sales_Lines columns:
['Month', 'CustomerID', 'CustomerName', 'Industry', 'SKU', 'Qty', 'Value (Rs)']


,Month,CustomerID,CustomerName,Industry,SKU,Qty,Value (Rs)
0,Apr-2026,C201,Customer 201,Cafe,SKU1011,96,21330
1,Apr-2026,C201,Customer 201,Cafe,SKU1014,37,11647
2,Apr-2026,C201,Customer 201,Cafe,SKU1016,45,6666



Customers columns:
['CustomerID', 'CustomerName', 'Industry', 'Area']


,CustomerID,CustomerName,Industry,Area
0,C201,Customer 201,Cafe,Zone 3
1,C202,Customer 202,Caterer,Zone 3
2,C203,Customer 203,Hotel,Zone 1


In [3]:

issues = []

def log_issue(problem, handling, source_fix):
    issues.append({
        "Problem": problem,
        "Handling": handling,
        "Recommended source fix": source_fix
    })

def clean_text(series):
    return (
        series.astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

# Clean text fields
for df, columns in [
    (items, [
        "SKU", "Item Name", "Category",
        "Alternate Item (name)", "Alternate Flag", "Status"
    ]),
    (stock, ["SKU"]),
    (sales, ["Month", "CustomerID", "CustomerName", "Industry", "SKU"]),
    (customers, ["CustomerID", "CustomerName", "Industry", "Area"])
]:
    for col in columns:
        df[col] = clean_text(df[col])

# Convert numeric fields
numeric_columns = {
    "items": (items, ["Master Pack Qty", "Unit Cost (Rs)"]),
    "stock": (stock, [
        "Qty 0-30 days", "Qty 31-60 days", "Qty 61-90 days",
        "Qty 91-180 days", "Qty 180+ days"
    ]),
    "sales": (sales, ["Qty", "Value (Rs)"])
}

for name, (df, columns) in numeric_columns.items():
    for col in columns:
        original = df[col].copy()
        converted = pd.to_numeric(original, errors="coerce")

        bad = original.notna() & converted.isna()

        if bad.any():
            log_issue(
                f"{name}: invalid numbers in {col} ({int(bad.sum())} rows)",
                "Invalid values changed to missing for review.",
                "Correct the field to a valid numeric value."
            )

        df[col] = converted

# Identify missing identifiers
for df, col, name in [
    (items, "SKU", "Item_Master"),
    (stock, "SKU", "Stock_Aging"),
    (sales, "SKU", "Sales_Lines"),
    (sales, "CustomerID", "Sales_Lines"),
    (customers, "CustomerID", "Customers")
]:
    count = int(df[col].isna().sum())
    if count:
        log_issue(
            f"{name}: missing {col} ({count} rows)",
            "Records cannot be reliably matched without an identifier.",
            f"Provide a valid {col} in the source system."
        )

# Check duplicate master and customer keys.
for df, key, name in [
    (items, "SKU", "Item_Master"),
    (customers, "CustomerID", "Customers")
]:
    duplicate_count = int(df.duplicated(key, keep=False).sum())
    if duplicate_count:
        log_issue(
            f"{name}: duplicate {key} ({duplicate_count} rows involved)",
            "Duplicates are reported; they need investigation.",
            f"Maintain unique {key} values."
        )

# Check duplicate stock SKUs, then aggregate stock buckets.
if stock["SKU"].duplicated(keep=False).any():
    log_issue(
        "Stock_Aging contains repeated SKUs",
        "Stock quantities are summed by SKU.",
        "Confirm whether repeated records are valid separate stock lots."
    )

stock_columns = [
    "Qty 0-30 days",
    "Qty 31-60 days",
    "Qty 61-90 days",
    "Qty 91-180 days",
    "Qty 180+ days"
]

# Missing stock buckets are assumed to be zero, but are flagged.
for col in stock_columns:
    count = int(stock[col].isna().sum())
    if count:
        log_issue(
            f"Missing stock bucket values in {col}: {count} rows",
            "Missing bucket values are treated as zero.",
            "Provide a complete stock-aging breakdown."
        )
    stock[col] = stock[col].fillna(0)

# Missing sales quantity/value is flagged and treated as zero for calculations.
for col in ["Qty", "Value (Rs)"]:
    count = int(sales[col].isna().sum())
    if count:
        log_issue(
            f"Missing sales values in {col}: {count} rows",
            "Missing values are treated as zero in calculations.",
            "Correct the missing sales quantity or value at source."
        )
    sales[col] = sales[col].fillna(0)

# Flag negative quantities, values, costs or stock.
for df, cols, name in [
    (stock, stock_columns, "Stock_Aging"),
    (sales, ["Qty", "Value (Rs)"], "Sales_Lines"),
    (items, ["Master Pack Qty", "Unit Cost (Rs)"], "Item_Master")
]:
    for col in cols:
        count = int((df[col] < 0).sum())
        if count:
            log_issue(
                f"{name}: negative values in {col} ({count} rows)",
                "Flagged for investigation; not silently converted.",
                "Confirm returns, adjustments or incorrect entries."
            )

# Check item master values
for col in ["Master Pack Qty", "Unit Cost (Rs)", "Category", "Status"]:
    count = int(items[col].isna().sum())
    if count:
        log_issue(
            f"Item_Master: missing {col} ({count} rows)",
            "Affected records are flagged for review.",
            f"Complete {col} in the item master."
        )

print("Data cleaning and checks completed.")


Data cleaning and checks completed.


Clean the data and record data-quality problems

In [4]:

def normalize_name(value):
    if pd.isna(value):
        return None
    return re.sub(r"\s+", " ", str(value)).strip().casefold()

# Build item-name-to-SKU mapping
name_to_sku = {}

for _, row in items.dropna(subset=["Item Name"]).iterrows():
    key = normalize_name(row["Item Name"])
    if key in name_to_sku and name_to_sku[key] != row["SKU"]:
        log_issue(
            f"Ambiguous item name: {row['Item Name']}",
            "Automatic alternate matching may be unreliable.",
            "Maintain unique item names or use an alternate SKU field."
        )
    else:
        name_to_sku[key] = row["SKU"]

items["Alternate SKU"] = items["Alternate Item (name)"].map(
    normalize_name
).map(name_to_sku)

# Identify old SKU -> replacement SKU
switch_map = {}

for _, row in items.iterrows():
    flag = str(row["Alternate Flag"]).strip().upper()
    old_sku = row["SKU"]
    new_sku = row["Alternate SKU"]

    if flag == "SWITCH":
        if pd.isna(new_sku):
            log_issue(
                f"Replacement item not found for {old_sku}",
                "Switch not applied because the replacement is unknown.",
                "Correct Alternate Item (name) in Item_Master."
            )
        elif old_sku == new_sku:
            log_issue(
                f"Item refers to itself as its replacement: {old_sku}",
                "Switch not applied.",
                "Correct the alternate item reference."
            )
        else:
            switch_map[old_sku] = new_sku

def canonical_sku(sku):
    seen = set()

    while sku in switch_map:
        if sku in seen:
            raise ValueError(f"Switch cycle detected at {sku}")

        seen.add(sku)
        sku = switch_map[sku]

    return sku

items["Canonical SKU"] = items["SKU"].map(canonical_sku)
stock["Canonical SKU"] = stock["SKU"].map(canonical_sku)
sales["Canonical SKU"] = sales["SKU"].map(canonical_sku)

print("Pack-size switches:")
print(switch_map if switch_map else "No switches detected.")

# Identify stock and sales SKUs absent from the item master.
known_skus = set(items["SKU"].dropna())

for df, name in [(stock, "Stock_Aging"), (sales, "Sales_Lines")]:
    unknown = sorted(set(df["SKU"].dropna()) - known_skus)

    if unknown:
        log_issue(
            f"Unknown SKUs in {name}: {unknown}",
            "These records cannot be matched to a valid item master record.",
            "Correct SKU codes or add the missing items to Item_Master."
        )

# Identify inactive stock using original SKU status.
inactive_skus = set(
    items.loc[
        items["Status"].str.casefold().eq("inactive").fillna(False),
        "SKU"
    ]
)

inactive_stock = stock[stock["SKU"].isin(inactive_skus)].copy()
inactive_stock["Total Stock"] = inactive_stock[stock_columns].sum(axis=1)

inactive_holding = inactive_stock[
    inactive_stock["Total Stock"] > 0
].copy()

print("\nInactive items still holding stock:")
display(inactive_holding)

# Active analysis excludes inactive items and switched-out old SKUs.
switched_out = set(switch_map.keys())

active_items = items[
    items["Status"].str.casefold().eq("active").fillna(False)
    & ~items["SKU"].isin(switched_out)
].copy()

print("\nActive replacement/master items:", len(active_items))


Pack-size switches:
{'SKU1003': 'SKU1006', 'SKU1015': 'SKU1019', 'SKU1030': 'SKU1033'}

Inactive items still holding stock:


,SKU,Qty 0-30 days,Qty 31-60 days,Qty 61-90 days,Qty 91-180 days,Qty 180+ days,Canonical SKU,Total Stock
40,SKU1041,59,34,25,31,20,SKU1041,169



Active replacement/master items: 38


Handle pack-size switches and inactive items

In [5]:

# Aggregate stock under the replacement SKU.
stock_agg = stock.groupby("Canonical SKU")[stock_columns].sum()

# Aggregate six-month sales under the replacement SKU.
sales_agg = sales.groupby("Canonical SKU").agg(
    Six_Month_Sales_Qty=("Qty", "sum"),
    Six_Month_Sales_Value=("Value (Rs)", "sum")
)

# Start with the replacement item's master pack and unit cost.
analysis = active_items.set_index("SKU").copy()

analysis = analysis.join(stock_agg, how="left")
analysis = analysis.join(sales_agg, how="left")

analysis[stock_columns] = analysis[stock_columns].fillna(0)

for col in ["Six_Month_Sales_Qty", "Six_Month_Sales_Value"]:
    analysis[col] = analysis[col].fillna(0)

# Rule R3: aged stock is the 91-180 and 180+ day buckets.
analysis["Total Stock"] = analysis[stock_columns].sum(axis=1)

analysis["Aged Quantity"] = (
    analysis["Qty 91-180 days"] + analysis["Qty 180+ days"]
)

analysis["Aged Value (Rs)"] = (
    analysis["Aged Quantity"] * analysis["Unit Cost (Rs)"]
)

# Rule R4: average monthly sales and months to clear.
analysis["Average Monthly Sales"] = (
    analysis["Six_Month_Sales_Qty"] / 6
)

analysis["Months to Clear"] = np.where(
    analysis["Average Monthly Sales"] > 0,
    analysis["Total Stock"] / analysis["Average Monthly Sales"],
    np.inf
)

# Flag records with missing or invalid pack/cost.
invalid_pack = (
    analysis["Master Pack Qty"].isna()
    | (analysis["Master Pack Qty"] <= 0)
)
invalid_cost = (
    analysis["Unit Cost (Rs)"].isna()
    | (analysis["Unit Cost (Rs)"] < 0)
)

for sku in analysis.index[invalid_pack]:
    log_issue(
        f"Invalid master pack quantity for {sku}",
        "Classified as Review Data until corrected.",
        "Provide a valid positive Master Pack Qty."
    )

for sku in analysis.index[invalid_cost]:
    log_issue(
        f"Invalid unit cost for {sku}",
        "Aged value may be unavailable or unreliable.",
        "Provide a valid non-negative Unit Cost (Rs)."
    )

# Rule R5: classification order matters.
def classify_item(row):
    pack = row["Master Pack Qty"]

    if pd.isna(pack) or pack <= 0:
        return "Review Data"

    if row["Six_Month_Sales_Qty"] < pack:
        return "Red List"

    if pd.isna(row["Unit Cost (Rs)"]) or row["Unit Cost (Rs)"] < 0:
        return "Review Data"

    if row["Aged Quantity"] <= 0:
        return "No Action"

    if row["Months to Clear"] < 3:
        return "No Action"

    return "Push"

analysis["Class"] = analysis.apply(classify_item, axis=1)

# Red List items are listed separately and receive no targets.
item_list = analysis.reset_index().rename(columns={"index": "SKU"})

print("Classification completed.")
display(
    item_list[
        ["SKU", "Item Name", "Category", "Total Stock",
         "Aged Quantity", "Aged Value (Rs)",
         "Six_Month_Sales_Qty", "Months to Clear", "Class"]
    ]
)


Classification completed.


,SKU,Item Name,Category,Total Stock,Aged Quantity,Aged Value (Rs),Six_Month_Sales_Qty,Months to Clear,Class
0,SKU1001,A-Product 01 (50/case),Category A,430,80,2000,853.0,3.024619,Push
1,SKU1002,A-Product 02 (10/case),Category A,37,11,935,3.0,74.000000,Red List
2,SKU1004,A-Product 04 (12/case),Category A,96,0,0,638.0,0.902821,No Action
3,SKU1005,A-Product 05 (50/case),Category A,306,92,31280,256.0,7.171875,Push
4,SKU1006,A-Product 06 (10/case),Category A,203,61,9150,306.0,3.980392,Push
5,SKU1007,A-Product 07 (10/case),Category A,92,0,0,604.0,0.913907,No Action
6,SKU1008,B-Product 01 (10/case),Category B,17,5,300,2.0,51.000000,Red List
7,SKU1009,B-Product 02 (12/case),Category B,72,22,7480,93.0,4.645161,Push
8,SKU1010,B-Product 03 (25/case),Category B,151,45,810,183.0,4.950820,Push
9,SKU1011,B-Product 04 (25/case),Category B,345,103,6180,501.0,4.131737,Push


Calculate stock, sales and item classifications

In [6]:

# Total spend per customer over the six-month period.
customer_spend = sales.groupby("CustomerID")["Value (Rs)"].sum()

# Build customer details from the customer master.
customer_info = customers.drop_duplicates(
    "CustomerID", keep="first"
).set_index("CustomerID")

# Fill missing customer names/industries from sales data when available.
sales_info = sales.groupby("CustomerID").agg(
    Sales_Name=("CustomerName", "first"),
    Sales_Industry=("Industry", "first")
)

customer_info = customer_info.join(sales_info, how="outer")

customer_info["CustomerName"] = customer_info["CustomerName"].fillna(
    customer_info["Sales_Name"]
)
customer_info["Industry"] = customer_info["Industry"].fillna(
    customer_info["Sales_Industry"]
)

# Purchase history, including both sides of a pack-size switch.
bought = sales[
    (sales["Qty"] > 0) | (sales["Value (Rs)"] > 0)
].copy()

customer_items = bought.groupby("CustomerID")["Canonical SKU"].apply(
    lambda values: set(values.dropna())
)

item_buyers = {}

for customer_id, sku_set in customer_items.items():
    for sku in sku_set:
        item_buyers.setdefault(sku, set()).add(customer_id)

# Eligible customers: strictly greater than Rs 1,00,000.
eligible_customers = set(
    customer_spend[customer_spend > 100000].index
)

# Categories purchased by each customer.
category_by_sku = analysis["Category"].to_dict()
customer_categories = {}

for customer_id, sku_set in customer_items.items():
    customer_categories[customer_id] = {
        category_by_sku[sku]
        for sku in sku_set
        if sku in category_by_sku and pd.notna(category_by_sku[sku])
    }

target_rows = []

for sku, item in analysis[analysis["Class"] == "Push"].iterrows():

    buyers = item_buyers.get(sku, set())

    buyer_industries = set()

    for cid in buyers:
        industry = customer_info["Industry"].get(cid, pd.NA)

        if pd.notna(industry) and str(industry).strip():
            buyer_industries.add(str(industry).strip().casefold())

    for cid in eligible_customers:
        industry = customer_info["Industry"].get(cid, pd.NA)

        industry_key = (
            str(industry).strip().casefold()
            if pd.notna(industry) and str(industry).strip()
            else None
        )

        if cid in buyers:
            bucket = 1
            bucket_name = "Bought before"

        elif (
            industry_key is not None
            and industry_key in buyer_industries
        ):
            bucket = 2
            bucket_name = "Similar business"

        elif item["Category"] in customer_categories.get(cid, set()):
            bucket = 3
            bucket_name = "Same category"

        else:
            continue

        target_rows.append({
            "SKU": sku,
            "Item Name": item["Item Name"],
            "Category": item["Category"],
            "CustomerID": cid,
            "Customer Name": customer_info["CustomerName"].get(cid, cid),
            "Industry": industry if pd.notna(industry) else "",
            "Area": customer_info["Area"].get(cid, ""),
            "Bucket": bucket,
            "Bucket Name": bucket_name,
            "Six-Month Purchase Value (Rs)": customer_spend.get(cid, 0)
        })

targets = pd.DataFrame(target_rows)

target_columns = [
    "SKU", "Item Name", "Category", "CustomerID", "Customer Name",
    "Industry", "Area", "Bucket", "Bucket Name",
    "Six-Month Purchase Value (Rs)"
]

if targets.empty:
    targets = pd.DataFrame(columns=target_columns + ["Call Priority"])
else:
    # One row per customer per item; bucket 1 wins if applicable.
    targets = targets.sort_values(
        ["SKU", "Bucket", "Six-Month Purchase Value (Rs)"],
        ascending=[True, True, False]
    )

    targets = targets.drop_duplicates(
        ["SKU", "CustomerID"], keep="first"
    )

    targets["Call Priority"] = (
        targets.groupby("SKU").cumcount() + 1
    )

print("Customer targeting completed.")
print("Eligible customers:", len(eligible_customers))
print("Customer target records:", len(targets))
display(targets.head(20))


Customer targeting completed.
Eligible customers: 14
Customer target records: 172


,SKU,Item Name,Category,CustomerID,Customer Name,Industry,Area,Bucket,Bucket Name,Six-Month Purchase Value (Rs),Call Priority
7,SKU1001,A-Product 01 (50/case),Category A,C220,Customer 220,Caterer,Zone 2,1,Bought before,689088,1
1,SKU1001,A-Product 01 (50/case),Category A,C224,Customer 224,Restaurant,Zone 1,1,Bought before,320780,2
0,SKU1001,A-Product 01 (50/case),Category A,C226,Customer 226,Caterer,Zone 3,1,Bought before,106498,3
5,SKU1001,A-Product 01 (50/case),Category A,C214,Customer 214,Caterer,Zone 1,2,Similar business,574001,4
6,SKU1001,A-Product 01 (50/case),Category A,C208,Customer 208,Caterer,Zone 2,2,Similar business,502556,5
3,SKU1001,A-Product 01 (50/case),Category A,C202,Customer 202,Caterer,Zone 3,2,Similar business,428435,6
2,SKU1001,A-Product 01 (50/case),Category A,C206,Customer 206,Restaurant,Zone 1,2,Similar business,253883,7
4,SKU1001,A-Product 01 (50/case),Category A,C221,Customer 221,Hotel,Zone 2,3,Same category,209491,8
8,SKU1001,A-Product 01 (50/case),Category A,C227,Customer 227,Hotel,Zone 2,3,Same category,182264,9
16,SKU1005,A-Product 05 (50/case),Category A,C220,Customer 220,Caterer,Zone 2,1,Bought before,689088,1


This cell applies Rule R6 and Rule R7. Only customers with total six-month purchase value greater than ₹1,00,000 qualify. Each customer receives only one bucket per Push item.

In [7]:

# Count items by class.
class_order = ["Red List", "No Action", "Push", "Review Data"]

class_summary = (
    analysis["Class"]
    .value_counts()
    .reindex(class_order, fill_value=0)
    .rename_axis("Class")
    .reset_index(name="Number of Items")
)

# Total aged value for Push items only.
push_aged_value = analysis.loc[
    analysis["Class"] == "Push", "Aged Value (Rs)"
].sum(min_count=1)

if pd.isna(push_aged_value):
    push_aged_value = 0

# Top 10 Push items by aged value.
top10 = (
    analysis[analysis["Class"] == "Push"]
    .sort_values("Aged Value (Rs)", ascending=False)
    .head(10)
    .reset_index()
)

display(Markdown("# Company X Slow-Stock Dashboard"))
display(Markdown(
    "**Stock date:** 30 September 2026  \n"
    "**Sales period:** April–September 2026"
))

print("\n1. NUMBER OF ITEMS IN EACH CLASS")
display(class_summary)

print(f"\n2. TOTAL AGED VALUE OF PUSH ITEMS: ₹{push_aged_value:,.2f}")

print("\n3. TOP 10 PUSH ITEMS BY AGED VALUE")
display(
    top10[
        ["SKU", "Item Name", "Category", "Total Stock",
         "Aged Quantity", "Aged Value (Rs)",
         "Six_Month_Sales_Qty", "Months to Clear", "Class"]
    ]
)

# Explain the classification of the three required SKUs.
print("\n4. REQUIRED SKU CLASSIFICATIONS")

for sku in ["SKU1001", "SKU1006", "SKU1019"]:
    if sku not in analysis.index:
        print(f"{sku}: Not found in active analysis.")
        continue

    row = analysis.loc[sku]

    if row["Class"] == "Red List":
        reason = (
            "Six-month sales are less than one master pack."
        )
    elif row["Class"] == "No Action":
        if row["Aged Quantity"] <= 0:
            reason = "There is no aged stock."
        else:
            reason = (
                "Months to clear are less than 3."
            )
    elif row["Class"] == "Push":
        reason = (
            "Sales are at least one master pack, aged stock exists, "
            "and months to clear are at least 3."
        )
    else:
        reason = "Required master data needs correction."

    print(
        f"{sku}: {row['Class']} | {reason} "
        f"| Sales: {row['Six_Month_Sales_Qty']} "
        f"| Pack: {row['Master Pack Qty']} "
        f"| Aged Qty: {row['Aged Quantity']} "
        f"| Months to clear: {row['Months to Clear']}"
    )

# First three customer targets for SKU1005.
print("\n5. FIRST THREE CUSTOMERS TO CALL FOR SKU1005")

sku1005 = targets[targets["SKU"] == "SKU1005"].head(3)

if sku1005.empty:
    print("No eligible targets found for SKU1005.")
else:
    display(
        sku1005[
            ["Call Priority", "CustomerID", "Customer Name",
             "Industry", "Area", "Bucket", "Bucket Name",
             "Six-Month Purchase Value (Rs)"]
        ]
    )


# Company X Slow-Stock Dashboard

**Stock date:** 30 September 2026  
**Sales period:** April–September 2026


1. NUMBER OF ITEMS IN EACH CLASS


,Class,Number of Items
0,Red List,7
1,No Action,12
2,Push,19
3,Review Data,0



2. TOTAL AGED VALUE OF PUSH ITEMS: ₹165,841.00

3. TOP 10 PUSH ITEMS BY AGED VALUE


,SKU,Item Name,Category,Total Stock,Aged Quantity,Aged Value (Rs),Six_Month_Sales_Qty,Months to Clear,Class
0,SKU1005,A-Product 05 (50/case),Category A,306,92,31280,256.0,7.171875,Push
1,SKU1034,E-Product 06 (100/case),Category E,797,239,20315,581.0,8.230637,Push
2,SKU1020,C-Product 06 (100/case),Category C,511,153,18360,479.0,6.400835,Push
3,SKU1019,C-Product 05 (12/case),Category C,907,210,12600,1310.0,4.154198,Push
4,SKU1029,E-Product 01 (25/case),Category E,379,113,9605,514.0,4.424125,Push
5,SKU1006,A-Product 06 (10/case),Category A,203,61,9150,306.0,3.980392,Push
6,SKU1033,E-Product 05 (10/case),Category E,455,137,8220,661.0,4.130106,Push
7,SKU1014,B-Product 07 (24/case),Category B,301,90,7650,555.0,3.254054,Push
8,SKU1009,B-Product 02 (12/case),Category B,72,22,7480,93.0,4.645161,Push
9,SKU1038,F-Product 03 (24/case),Category F,204,61,7320,171.0,7.157895,Push



4. REQUIRED SKU CLASSIFICATIONS
SKU1001: Push | Sales are at least one master pack, aged stock exists, and months to clear are at least 3. | Sales: 853.0 | Pack: 50 | Aged Qty: 80 | Months to clear: 3.0246189917936697
SKU1006: Push | Sales are at least one master pack, aged stock exists, and months to clear are at least 3. | Sales: 306.0 | Pack: 10 | Aged Qty: 61 | Months to clear: 3.980392156862745
SKU1019: Push | Sales are at least one master pack, aged stock exists, and months to clear are at least 3. | Sales: 1310.0 | Pack: 12 | Aged Qty: 210 | Months to clear: 4.154198473282443

5. FIRST THREE CUSTOMERS TO CALL FOR SKU1005


,Call Priority,CustomerID,Customer Name,Industry,Area,Bucket,Bucket Name,Six-Month Purchase Value (Rs)
16,1,C220,Customer 220,Caterer,Zone 2,1,Bought before,689088
10,2,C224,Customer 224,Restaurant,Zone 1,1,Bought before,320780
14,3,C214,Customer 214,Caterer,Zone 1,2,Similar business,574001


Display the case study results

In [8]:

!pip -q install ipywidgets

import ipywidgets as widgets
from IPython.display import display, clear_output

item_list = analysis.reset_index()

class_options = ["All"] + sorted(
    item_list["Class"].dropna().unique().tolist()
)

category_options = ["All"] + sorted(
    item_list["Category"].dropna().unique().tolist()
)

class_filter = widgets.Dropdown(
    options=class_options,
    value="All",
    description="Class:"
)

category_filter = widgets.Dropdown(
    options=category_options,
    value="All",
    description="Category:"
)

output = widgets.Output()

def filter_items(change=None):
    with output:
        clear_output(wait=True)

        result = item_list.copy()

        if class_filter.value != "All":
            result = result[
                result["Class"] == class_filter.value
            ]

        if category_filter.value != "All":
            result = result[
                result["Category"] == category_filter.value
            ]

        display(result)

class_filter.observe(filter_items, names="value")
category_filter.observe(filter_items, names="value")

display(widgets.HBox([class_filter, category_filter]))
display(output)

filter_items()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 59.7 MB/s eta 0:00:00


Output()

Filter items by class and category - This creates interactive dropdown filters inside Colab. It is useful for checking the item list, although it is not yet a permanently hosted web app.

In [9]:

quality_report = pd.DataFrame(issues).drop_duplicates()

print("DATA QUALITY REPORT")

if quality_report.empty:
    print("No configured data-quality issues were detected.")
else:
    display(quality_report)

print("\nINACTIVE ITEMS WITH STOCK")
display(inactive_holding)


DATA QUALITY REPORT


,Problem,Handling,Recommended source fix
0,Stock_Aging: negative values in Qty 31-60 days...,Flagged for investigation; not silently conver...,"Confirm returns, adjustments or incorrect entr..."
1,"Unknown SKUs in Sales_Lines: ['sku1001', 'sku1...",These records cannot be matched to a valid ite...,Correct SKU codes or add the missing items to ...



INACTIVE ITEMS WITH STOCK


,SKU,Qty 0-30 days,Qty 31-60 days,Qty 61-90 days,Qty 91-180 days,Qty 180+ days,Canonical SKU,Total Stock
40,SKU1041,59,34,25,31,20,SKU1041,169


Show all data-quality problems

In [10]:

from google.colab import files

output_file = "Company_X_Slow_Stock_Review.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    class_summary.to_excel(
        writer, sheet_name="Class_Summary", index=False
    )

    item_list.to_excel(
        writer, sheet_name="Item_Analysis", index=False
    )

    top10.to_excel(
        writer, sheet_name="Top_10_Push", index=False
    )

    targets.to_excel(
        writer, sheet_name="Customer_Targets", index=False
    )

    quality_report.to_excel(
        writer, sheet_name="Data_Quality", index=False
    )

    inactive_holding.to_excel(
        writer, sheet_name="Inactive_Stock", index=False
    )

    # Include the original source sheets for traceability.
    items.to_excel(
        writer, sheet_name="Item_Master_Cleaned", index=False
    )

    stock.to_excel(
        writer, sheet_name="Stock_Aging_Cleaned", index=False
    )

    sales.to_excel(
        writer, sheet_name="Sales_Lines_Cleaned", index=False
    )

    customers.to_excel(
        writer, sheet_name="Customers_Cleaned", index=False
    )

print("Excel report created successfully.")
files.download(output_file)


Excel report created successfully.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>